### Récupération des liens Google Scholar

In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import pandas as pd

In [3]:
#Mise en place du driver
chrome_options = webdriver.ChromeOptions()
driver = webdriver.Chrome(options=chrome_options)
wait = WebDriverWait(driver, 200)

In [4]:
#Mise en place de la requête et de la date de début et fin des articles à explorer
query = "reinforcement learning for trading"
start, end = 2020, 2025
url = f"https://scholar.google.com/scholar?hl=fr&q={query}&as_ylo={start}&as_yhi={end}"
driver.get(url)

#Attendre que les données charges
time.sleep(2)

Exploration des pages de résultats

In [5]:
#Mise en place d'une attente de 40 secondes pour pouvoir répondre aux captchas
time.sleep(40)

#Récupérer tous les éléments <a> à savoir les liens
links = driver.find_elements(By.TAG_NAME, "a")

#On initialise une liste titres et liens qui contiendrons les informations éponymes
titres=[]
liens=[]
for link in links:
    text = link.text.strip()
    href = link.get_attribute("href")
    #Définition des conditions d'inclusion des articles
    if (text and href) and (("reinforcement learning" in text.lower() and "trading" in text.lower())) :
        print(f"{text} -> {href}")
        titres.append(text)
        liens.append(href)

suivant=True
#Si bouton suivant présent, alors on clique dessus et on extrait les autres articles
while suivant==True:

    ##Récupérer tous les éléments <a> à savoir les liens
    links = driver.find_elements(By.TAG_NAME, "a")

    for link in links:
        text = link.text.strip()
        href = link.get_attribute("href")
        #Définition des conditions d'inclusion des articles
        if (text and href) and ("reinforcement learning" in text.lower() and "trading" in text.lower()) :
            print(f"{text} -> {href}")
            titres.append(text)
            liens.append(href)
    #On essaie de cliquer sur le bouton suivant s'il est présent /!\ Modifier "suivant" si vous utiliser google dans une autre langue
    try:
        next_btn = driver.find_element(By.LINK_TEXT, "Suivant")
        next_btn.click()
        time.sleep(2)
    except:
        #Sinon c'est qu'il n'y plus de lien "suivant"
            print("Fin des pages : 'Suivant' n'est plus un lien.")
            suivant=False


Reinforcement learning for quantitative trading -> https://dl.acm.org/doi/abs/10.1145/3582560
Deep reinforcement learning in quantitative algorithmic trading: A review -> https://arxiv.org/abs/2106.00123
Adaptive quantitative trading: An imitative deep reinforcement learning approach -> https://aaai.org/ojs/index.php/AAAI/article/view/5587
FinRL: Deep reinforcement learning framework to automate trading in quantitative finance -> https://dl.acm.org/doi/abs/10.1145/3490354.3494366
Adaptive stock trading strategies with deep reinforcement learning methods -> https://www.sciencedirect.com/science/article/pii/S0020025520304692
An application of deep reinforcement learning to algorithmic trading -> https://www.sciencedirect.com/science/article/pii/S0957417421000737
Deep reinforcement learning approach for trading automation in the stock market -> https://ieeexplore.ieee.org/abstract/document/9877940/
FinRL: A deep reinforcement learning library for automated stock trading in quantitative fi

In [6]:
len(titres)

44

Enregistrement des articles dans un csv

In [7]:
df=pd.DataFrame({'Titres':titres,'liens':liens})
df.to_csv("Liste_articles.csv",index=False, sep=";")

driver.quit()

### Lecture des abstracts des articles

In [8]:
articles=pd.read_csv("Liste_articles.csv", sep=";")
print(articles.shape)

(44, 2)


Suppression des doublons

In [9]:
articles.duplicated().sum()

np.int64(22)

Suppression des doublons

In [10]:
articles=articles.drop_duplicates(subset=["Titres","liens"], keep="first")
print(articles.shape)

(22, 2)


In [11]:
articles.duplicated().sum()

np.int64(0)

Suppression des liens Google Scholar présent dans nos liens conservés

In [12]:
articles=articles[~articles["liens"].str.contains("https://scholar.google")]

In [13]:
articles.shape

(10, 2)

In [15]:
import requests
from PyPDF2 import PdfReader
import io
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException


options = webdriver.ChromeOptions()
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                     "AppleWebKit/537.36 (KHTML, like Gecko) "
                     "Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=chrome_options)

#Fonction permettant de lire approximativement les fichier pdf pour trouver l'abstract
def extract_pdf_abstract(url):
    response = requests.get(url)
    pdf_file = io.BytesIO(response.content)
    reader = PdfReader(pdf_file)
    abstract_text = ""
    for page in reader.pages[:10]: #On définit 10 comme ultime page à conserver de façon à trouver l'abstract dans les 10 premières pages 
        text = page.extract_text()
        if text and "Abstract" in text:
            abstract_text += text
            break
    return abstract_text.strip()

#Fonction permettant de lire les abstract pour des fichiers HTML
def extract_html_abstract(url):
    driver.get(url)
    time.sleep(3)
    abstract_text = None
    
    #Essayer de chercher un bouton pour accepter les cookies
    try:
        #l'ID des cookies est généralement onetrust, et on attent 5 secondes que le bouton arrive
        accept_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.ID, "onetrust-accept-btn-handler"))
        )
        accept_button.click()
    #Sinon, on essaie de trouver un bouton qui contient accepter, accept ou allow
    except TimeoutException:
        try:
           
            accept_button = WebDriverWait(driver, 5).until(
                EC.element_to_be_clickable((By.XPATH, "//button[contains(.,'Accept') or contains(.,'Accepter') or contains(.,'Allow')]"))
            )
            accept_button.click()
        except TimeoutException:
            print("Pas de bannière cookies détectée")
            
    # Essayer de cliquer sur un bouton "Abstract" s'il y'en a un présent, ce qui peut arriver
    try:
        abstract_button = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//*[contains(text(),'Abstract') or contains(text(),'abstract')]"))
        )
        abstract_button.click()
        time.sleep(2)
    #Sinon, on passe à l'étape suivante
    except TimeoutException:
        pass
    
    #Essayer plusieurs sélecteurs pour les abstracts
    selectors = [
        (By.ID, "abstract"),
        (By.CLASS_NAME, "abstract"),
        (By.XPATH, "//section[contains(@class,'abstract')]"),
        (By.XPATH, "//div[contains(@class,'abstract')]"),
        (By.XPATH, "//*[contains(text(),'Abstract')]/following::p[1]"),
    ]
    
    #Pour chaque sélécteur, on regarde si on trouve quelque chose et si c'est bien le cas, on extrait le texte de l'abstract
    for by, value in selectors:
        try:
            element = driver.find_element(by, value)
            abstract_text = element.text.strip()
            if abstract_text:
                break
        except NoSuchElementException:
            continue
    
    return abstract_text

#Mise en place de l'extraction d'abstract 
def get_abstract(url):
    #si le lien fini par ".PDF" alors on utilise la fonction extract_pdf_abstract
    if url.endswith(".pdf"):
        return extract_pdf_abstract(url)
    #sinon on utilise la fonction extract_html_abstract
    else:
        return extract_html_abstract(url)

#Mise en place de la recherche des abstracts
urls = articles["liens"]

#Exécution de la recherche
results = {}
for url in urls:
    abstract = get_abstract(url)
    results[url] = abstract
    print(f"URL: {url}")
    print(f"Abstract: {abstract}")




URL: https://dl.acm.org/doi/abs/10.1145/3582560
Abstract: Quantitative trading (QT), which refers to the usage of mathematical models and data-driven techniques in analyzing the financial market, has been a popular topic in both academia and financial industry since 1970s. In the last decade, reinforcement learning (RL) has garnered significant interest in many domains such as robotics and video games, owing to its outstanding ability on solving complex sequential decision making problems. RL’s impact is pervasive, recently demonstrating its ability to conquer many challenging QT tasks. It is a flourishing research direction to explore RL techniques’ potential on QT tasks. This paper aims at providing a comprehensive survey of research efforts on RL-based methods for QT tasks. More concretely, we devise a taxonomy of RL-based QT models, along with a comprehensive summary of the state of the art. Finally, we discuss current challenges and propose future research directions in this excit

In [16]:
results

{'https://dl.acm.org/doi/abs/10.1145/3582560': 'Quantitative trading (QT), which refers to the usage of mathematical models and data-driven techniques in analyzing the financial market, has been a popular topic in both academia and financial industry since 1970s. In the last decade, reinforcement learning (RL) has garnered significant interest in many domains such as robotics and video games, owing to its outstanding ability on solving complex sequential decision making problems. RL’s impact is pervasive, recently demonstrating its ability to conquer many challenging QT tasks. It is a flourishing research direction to explore RL techniques’ potential on QT tasks. This paper aims at providing a comprehensive survey of research efforts on RL-based methods for QT tasks. More concretely, we devise a taxonomy of RL-based QT models, along with a comprehensive summary of the state of the art. Finally, we discuss current challenges and propose future research directions in this exciting field.

In [17]:
len(results)

10

In [18]:
df = pd.DataFrame(list(results.items()), columns=["url", "abstract"])
df

,url,abstract
0,https://dl.acm.org/doi/abs/10.1145/3582560,"Quantitative trading (QT), which refers to the..."
1,https://arxiv.org/abs/2106.00123,Algorithmic stock trading has become a staple ...
2,https://aaai.org/ojs/index.php/AAAI/article/vi...,"Abstract\nIn recent years, considerable effort..."
3,https://dl.acm.org/doi/abs/10.1145/3490354.349...,None
4,https://www.sciencedirect.com/science/article/...,Highlights\n•\nGated Recurrent Unit is propose...
5,https://www.sciencedirect.com/science/article/...,None
6,https://ieeexplore.ieee.org/abstract/document/...,Abstract:\nDeep Reinforcement Learning (DRL) a...
7,https://arxiv.org/abs/2011.09607,As deep reinforcement learning (DRL) has been ...
8,https://dl.acm.org/doi/abs/10.1145/3383455.342...,None
9,https://link.springer.com/article/10.1007/s006...,The role of the stock market across the overal...


Nettoyage des données textuelles

In [19]:
import re
abstracts=[]
for t in df["abstract"]:
    if t!=None:
        cleaned = re.sub(r"^Abstract:\s*", "", t)  # supprime "Abstract:" au début + espaces/sauts de ligne
        cleaned = re.sub(r"^Abstract\s*", "", cleaned)# supprime "Abstract" au début + espaces/sauts de ligne
        cleaned = cleaned.replace("\n"," ")# supprime "\n" au début + espaces/sauts de ligne
        cleaned = re.sub(r"[^a-zA-Z0-9]", " ", cleaned)# suppression des caractère non alphanumérique

        abstracts.append(cleaned)
    else:
        abstracts.append("NA")

In [20]:
abstracts

['Quantitative trading  QT   which refers to the usage of mathematical models and data driven techniques in analyzing the financial market  has been a popular topic in both academia and financial industry since 1970s  In the last decade  reinforcement learning  RL  has garnered significant interest in many domains such as robotics and video games  owing to its outstanding ability on solving complex sequential decision making problems  RL s impact is pervasive  recently demonstrating its ability to conquer many challenging QT tasks  It is a flourishing research direction to explore RL techniques  potential on QT tasks  This paper aims at providing a comprehensive survey of research efforts on RL based methods for QT tasks  More concretely  we devise a taxonomy of RL based QT models  along with a comprehensive summary of the state of the art  Finally  we discuss current challenges and propose future research directions in this exciting field ',
 'Algorithmic stock trading has become a st

In [21]:
df["abstract"]=abstracts
df

,url,abstract
0,https://dl.acm.org/doi/abs/10.1145/3582560,Quantitative trading QT which refers to the...
1,https://arxiv.org/abs/2106.00123,Algorithmic stock trading has become a staple ...
2,https://aaai.org/ojs/index.php/AAAI/article/vi...,In recent years considerable efforts have bee...
3,https://dl.acm.org/doi/abs/10.1145/3490354.349...,NA
4,https://www.sciencedirect.com/science/article/...,Highlights Gated Recurrent Unit is proposed ...
5,https://www.sciencedirect.com/science/article/...,NA
6,https://ieeexplore.ieee.org/abstract/document/...,Deep Reinforcement Learning DRL algorithms c...
7,https://arxiv.org/abs/2011.09607,As deep reinforcement learning DRL has been ...
8,https://dl.acm.org/doi/abs/10.1145/3383455.342...,NA
9,https://link.springer.com/article/10.1007/s006...,The role of the stock market across the overal...


In [22]:
articles_google_scholar=df[df["abstract"]!="NA"]

Exraction des articles intéressants pour analyse d'abstract

In [23]:
articles_google_scholar.to_csv("articles_int_google_scholar.csv",sep=";",index=False)